# Find optimal parameters for UMAP

In [1]:
### Imports
from rail.core.data import DataStore
from rail.core.stage import RailStage
from rail.core.data import PqHandle

import numpy as np
import h5py
import pandas as pd
import tables_io

from UMAPEstimator import UMAPEstimator

from rail.evaluation.dist_to_point_evaluator import DistToPointEvaluator
import optuna

# Specify paths and parameters

In [2]:
### Specify path to noisy catalog
noisy_catalog_path = '/Users/leo/Projects/LBG_cosmology/simulated_catalogs/integrated_catalog_23apr26_noised_19Jun26.pq'

### Specify path to output photometry
import time
date = time.strftime('%d%b%y', time.localtime())
photoz_path = noisy_catalog_path.split('.pq')[0] + f'_UMAPphotoz_{date}.pq'

# Split data into training and validation

In [3]:
training_fraction = 0.8


metric            = "manhattan_weighted_linear"
seed              = 42

In [8]:
data = tables_io.read(noisy_catalog_path)

data_cut = int(1e3)
# data_cut = -1
data = data[:data_cut]

training_indices = np.zeros(len(data), dtype = bool)
training_indices[np.random.choice(len(data), size = int(training_fraction * len(data)),
                 replace = False)] = True

bands = [key for key in data.keys() if (not key.endswith('_err')) & (key != 'Roman_F146')]
error_bands = [key for key in data.keys() if key.endswith('_err')]

validation_data = data[~training_indices]
training_data   = data[training_indices]

column_list None


In [9]:
photometry_bands = [key for key in training_data.keys()\
                        if (not key.endswith('_err')) and (key != 'Roman_F146')]
phot_error_bands = [f"{key}_err" for key in photometry_bands]

In [10]:
redshift_filepath = '/Users/leo/Projects/LBG_cosmology/surveys/pop-cosmos/mock_catalog/mock_catalog_Ch1_26.h5'
redshift          = h5py.File(redshift_filepath)['sfh_parameters'][:, -1]

training_redshift   = redshift[:data_cut][training_indices]
validation_redshift = redshift[:data_cut][~training_indices]

# Instantiate and run RAIL stage

In [ ]:
RailStage.data_store = DataStore()

stage = UMAPEstimator.make_stage(
    name = "UMAP_informer",
    
    ambient_metric_umap = metric,
    
    seed = seed
)

stage.set_data("training_photometry", data = training_data[photometry_bands])
stage.set_data("training_phot_error", data = training_data[phot_error_bands])
stage.set_data("training_redshift",   data = training_redshift)

# Inform the estimator

In [ ]:
stage.UMAP_informer()

# Estimate redshifts

In [ ]:
stage.set_data("estimation_photometry", data = validation_data[photometry_bands])
stage.set_data("estimation_phot_error", data = validation_data[phot_error_bands])
stage.UMAP_estimator()

In [ ]:
estimated_photoz_pdfs = stage.get_handle('estimated_photoz_pdfs').data
estimated_photoz_medians = stage.get_handle('estimated_photoz_medians').data

# Evaluate

# Plot redshifts

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('/Users/leo/Projects/LBG_cosmology/code/umap_nz_cal.mplstyle')

In [ ]:
fig = plt.figure()
ax = fig.add_subplot()
ax.set_xlim(0, .5)

index = 0

estimated_photoz_pdfs.plot(index, axes = ax)
ax.axvline(estimated_photoz_medians.values[index])

ax.set_xlabel('Redshift')
ax.set_ylabel('Density')

In [ ]:
evaluator = DistToPointEvaluator.make_stage(
    name = "DistToPointEvaluator",
    metrics = ["cdeloss"],
    reference_dictionary_key = "true_z",
    hdf5_groupname = "",
    metric_integration_limits = [0, 3],
    dx = 0.001,
    output_mode = "return"
)

In [ ]:
evaluation = evaluator.evaluate(estimated_photoz_pdfs,
                                pd.DataFrame({"true_z": validation_redshift}))

In [14]:
### Hedging: load data
### Hedging: 

def objective(trial):
    
    RailStage.data_store = DataStore()
    
    n_neighbors_umap = trial.suggest_int("n_neighbors_umap", 10, 200, step  = 5)
    min_dist         = trial.suggest_float("min_dist",       0,  1,   step  = 0.05)
    
    n_neighbors_knn = trial.suggest_int("n_neighbors_knn",  5, 200, step  = 5)
    metric_p_knn    = trial.suggest_int("n_neighbors_umap", 1, 5,   step  = 1)
    

    estimator = UMAPEstimator.make_stage(
        name = "UMAP_estimator",
        
        ambient_metric_umap = metric,
        
        n_neighbors_umap = n_neighbors_umap,
        min_dist         = min_dist,
        
        n_neighbors_knn = n_neighbors_knn,
        metric_p_knn    = metric_p_knn,
        
        seed = seed
    )

    estimator.set_data("training_photometry", data = training_data[photometry_bands])
    estimator.set_data("training_phot_error", data = training_data[phot_error_bands])
    estimator.set_data("training_redshift",   data = training_redshift)
    
    estimator.UMAP_informer()
    
    estimator.set_data("estimation_photometry", data = validation_data[photometry_bands])
    estimator.set_data("estimation_phot_error", data = validation_data[phot_error_bands])
    estimator.UMAP_estimator()
    
    evaluator = DistToPointEvaluator.make_stage(
        name = "DistToPointEvaluator",
        metrics = ["cdeloss"],
        reference_dictionary_key = "true_z",
        hdf5_groupname = "",
        metric_integration_limits = [0, 3],
        dx = 0.001,
        output_mode = "return"
    )
    
    evaluation = evaluator.evaluate(estimator.get_handle('estimated_photoz_pdfs').data,
                                     pd.DataFrame({"true_z": validation_redshift}))
    
    cdeloss = evaluation["summary"].read()['cdeloss'][0]
    
    return cdeloss

In [ ]:
study = optuna.create_study()

[I 2026-06-25 16:20:52,092] A new study created in memory with name: no-name-d609dc8c-e872-42b5-8b12-74370ce5b46d


In [16]:
study.optimize(objective, n_trials = 10)

/var/folders/wj/mkr9rd213rjdty0rm0wjqqm00000gn/T/ipykernel_66206/2019679359.py:12: RuntimeWarning: Inconsistent parameter values for distribution with name "n_neighbors_umap"! This might be a configuration mistake. Optuna allows to call the same distribution with the same name more than once in a trial. When the parameter values are inconsistent optuna only uses the values of the first call and ignores all following. Using these values: {'log': False, 'step': 5, 'low': 10, 'high': 200}
  metric_p_knn    = trial.suggest_int("n_neighbors_umap", 1, 5,   step  = 1)
/Users/leo/miniforge3/envs/RAIL/lib/python3.14/site-packages/umap/umap_.py:1857: UserWarning: custom distance metric does not return gradient; inverse_transform will be unavailable. To enable using inverse_transform method, define a distance function that returns a tuple of (distance [float], gradient [np.array])
  warn(
/Users/leo/miniforge3/envs/RAIL/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 

Inserting handle into data store.  training_photometry: None, UMAP_estimator
Inserting handle into data store.  training_phot_error: None, UMAP_estimator
Inserting handle into data store.  training_redshift: None, UMAP_estimator
Inserting handle into data store.  informed_reducer_UMAP_estimator: inprogress_informed_reducer_UMAP_estimator.pkl, UMAP_estimator
Inserting handle into data store.  informed_embedding_UMAP_estimator: inprogress_informed_embedding_UMAP_estimator.pq, UMAP_estimator
Inserting handle into data store.  informed_kNN_regressor_UMAP_estimator: inprogress_informed_kNN_regressor_UMAP_estimator.pkl, UMAP_estimator
Inserting handle into data store.  estimation_photometry: None, UMAP_estimator
Inserting handle into data store.  estimation_phot_error: None, UMAP_estimator


[W 2026-06-25 16:20:55,410] Trial 0 failed with parameters: {'n_neighbors_umap': 130, 'min_dist': 0.75, 'n_neighbors_knn': 45} because of the following error: OSError('Unable to synchronously create file (unable to truncate a file which is already open)').
Traceback (most recent call last):
  File "/Users/leo/miniforge3/envs/RAIL/lib/python3.14/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/wj/mkr9rd213rjdty0rm0wjqqm00000gn/T/ipykernel_66206/2019679359.py", line 49, in objective
    evaluation = evaluator.evaluate(estimator.get_handle('estimated_photoz_pdfs').data,
                                     pd.DataFrame({"true_z": validation_redshift}))
  File "/Users/leo/miniforge3/envs/RAIL/lib/python3.14/site-packages/rail/evaluation/evaluator.py", line 126, in evaluate
    self.run()
    ~~~~~~~~^^
  File "/Users/leo/miniforge3/envs/RAIL/lib/python3.14/site-packages/rail/evaluation/evaluator.py", line 154, in run
 

Inserting handle into data store.  estimated_embedding_UMAP_estimator: inprogress_estimated_embedding_UMAP_estimator.pq, UMAP_estimator
Inserting handle into data store.  estimated_photoz_medians_UMAP_estimator: inprogress_estimated_photoz_medians_UMAP_estimator.pq, UMAP_estimator
Inserting handle into data store.  estimated_photoz_pdfs_UMAP_estimator: inprogress_estimated_photoz_pdfs_UMAP_estimator.hdf5, UMAP_estimator
Inserting handle into data store.  input: None, DistToPointEvaluator
Inserting handle into data store.  truth: None, DistToPointEvaluator
Requested metrics: ['cdeloss']
Processing 0 running evaluator on chunk 0 - 200.
Inserting handle into data store.  output_DistToPointEvaluator: inprogress_output_DistToPointEvaluator.hdf5, DistToPointEvaluator


OSError: Unable to synchronously create file (unable to truncate a file which is already open)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot()


ax.hexbin(validation_redshift, estimated_photoz_medians,
          mincnt = 1)
ax.axline((0, 0), slope = 1, color = 'red')

In [ ]:
def sigma_NMAD(estimated_z, true_z):
    
    delta_z = estimated_z - true_z
    
    return 1.4826 * np.median(np.abs(delta_z/(1 + true_z) - ))

In [ ]:
fig = plt.figure()
ax = fig.add_subplot()


ax.hexbin(validation_redshift, estimated_photoz_medians,
          mincnt = 1)
ax.axline((0, 0), slope = 1, color = 'red')

In [ ]:
# Save otputs